In [8]:
#read multiple excel sheets into a dataframe Final - preprocessed
#read each sheet 2 separate times given the column headers/data rows aren't on the same line
#then concat them axis=1, ignore_index=False, looping through all worksheets, clean the data,
#and finally export to csv
import pandas as pd
#pd.set_option('display.max_rows', 3000)
pd.reset_option('display.max_rows')
#pd.set_option('display.max_columns', 357)
pd.reset_option('display.max_columns')

import numpy as np
pd.set_option('future.no_silent_downcasting', True) #this keeps nullable int64 as is (no downcasting to a smaller INT32)
#!pip install xlrd>=2.0.1

excel_file_path = r'C:\Users\kedre\Documents\post_PPRI\MIT_xPRO_data_engineering_bootcamp\Solutions\mrtssales92-present.xls'

# Exclude 2021 only in testing
excluded_sheets = []

# Get all worksheet names
excel_file = pd.ExcelFile(excel_file_path)
all_sheet_names = excel_file.sheet_names

# Filter out sheet 2021 (only in testing) and sort remaining by year
sheet_names = sorted([sheet for sheet in all_sheet_names if sheet not in excluded_sheets])

# Initialize list to store the processed dataframes
processed_dfs = []

#--------------------------------------EXCEL SHEET PROCESSING BEGIN------------------------------------------------------
for sheet_name in sheet_names:
    try:
        #print(f"Processing sheet: {sheet_name}")
        
        #Read business column(s)
        business_cols_df = pd.read_excel(
            excel_file_path, 
            header=3, 
            sheet_name=sheet_name,
            skiprows=range(4,6),
            usecols='B' #'A:B' if NAICS CODE is ever needed
        )
        
        #Read data columns, if 2021, there are a couple differences in the data columns:
        #1. usecols is shorter range (C:E) than the other years (C:O)
        #2. "CY CUM" is the TOTAL column name instead of "TOTAL" like the other worksheets
        from_col = 'C'
        to_col = 'O' #the longer range of columns
        if sheet_name == '2021': 
            to_col='E' #the shorter range of columns
        column_range_string = f"{from_col}:{to_col}"
        data_cols_df = pd.read_excel(
            excel_file_path, 
            header=4, 
            sheet_name=sheet_name,
            skiprows=range(5,6),
            usecols=column_range_string
        )

        # change "CY CUM" column head to match "TOTAL" of the other worksheets if this worksheet is 2021
        if sheet_name == '2021': 
            data_cols_df = data_cols_df.rename(columns={'CY CUM': 'TOTAL'})

        #Remove year from data column names, e.g. "Jan. 2020", and replace month w/ month _#, e.g. _1
        month_mapping = {
            'jan': '_1', 'feb': '_2', 'mar': '_3', 'apr': '_4', 'may': '_5', 'jun': '_6',
            'jul': '_7', 'aug': '_8', 'sep': '_9', 'oct': '_10', 'nov': '_11', 'dec': '_12'
        }
        
        new_column_names = {}
        for col in data_cols_df.columns:
            # Extract month name by removing year and whatever extra
            clean_col = col.lower().replace('.', '').replace(' ', '')
            # Remove year (assuming it's at the end)
            for year_str in [sheet_name, str(sheet_name)]:
                clean_col = clean_col.replace(year_str.lower(), '')
            
            # Map month name to number
            for month_name, month_num in month_mapping.items():
                if month_name in clean_col:
                    new_column_names[col] = month_num
                    break
            else:
                # If no month found, keep original column name
                new_column_names[col] = col
        
        data_cols_df.rename(columns=new_column_names, inplace=True)
        
        # Combine the dataframes
        sheet_df = pd.concat([business_cols_df, data_cols_df], axis=1, ignore_index=False)
        
        # Add Year column (converting sheet name to integer)
        try:
            year_value = int(sheet_name)
            sheet_df['Year'] = year_value
        except ValueError:
            print(f"Warning: Could not convert sheet name '{sheet_name}' to integer. Using sheet name as string.")
            sheet_df['Year'] = sheet_name
        
        # Filter to the 'NOT ADJUSTED' only
        trgt = 'ADJUSTED'
        trgt_col = 'Kind of Business'
        try:
            wheres_adjusted = sheet_df[sheet_df[trgt_col].str.contains(trgt, na=False)].index[0]
            sheet_df_unadjusted = sheet_df.iloc[:wheres_adjusted]
        except IndexError:
            print(f"The value '{trgt}' was not found in column '{trgt_col}' for sheet '{sheet_name}'. Using all rows.")
            sheet_df_unadjusted = sheet_df
        except KeyError:
            print(f"Column '{trgt_col}' not found in sheet '{sheet_name}'. Using all rows.")
            sheet_df_unadjusted = sheet_df
        
        # Add processed dataframe to list
        processed_dfs.append(sheet_df_unadjusted)
        
    except Exception as e:
        print(f"Error processing sheet '{sheet_name}': {str(e)}")
        continue

#combine all processed dataframes
if processed_dfs:
    all_sheets_df_combined = pd.concat(processed_dfs, axis=0, ignore_index=True)
    print(f"Successfully combined {len(processed_dfs)} sheets into a single dataframe.")
    print(f"Combined dataframe shape: {all_sheets_df_combined.shape}")
    
    # Debug info
    #print("\nFirst 5 rows:")
    #print(all_sheets_df_combined.head())
    #print(f"\nColumns: {list(all_sheets_df_combined.columns)}")
    #print(f"Years included: {sorted(all_sheets_df_combined['Year'].unique())}")
else:
    print("No dataframes were successfully processed.")
    all_sheets_df_combined = pd.DataFrame()
#--------------------------------------EXCEL SHEET PROCESSING END------------------------------------------------------

#If there is time, get a list of row names X %null/missing, then clean the final dataframe going away with those rows more carefully
#handled in time series analysis.  Also, pandas.melt() SEE Module8_scratch!!!!!  Do that in the time series analysis code or here?  There are a couple 
#values that are, by definition in the data itself, null:  (NA), and 
#(S).  The former is null.  (S) =  Suppressed - "does not meet pub standards"  See also Horvitz-Thompson estimates
#in https://www.census.gov/retail/mrts/www/benchmark/2022/pdf/Introduction.pdf.  Is there a Python function that 
#uses the Horvitz-Thompson estimate, if relevant in our analysis?  None found yet.
#embedded spaces in column name aren't allowed in MySQL and for most efficient INSERT, the table and CSV column
#names have to match
all_sheets_df_combined = all_sheets_df_combined.rename(columns={'Kind of Business': 'business_type'})

#get the columns that should be INT (numeric) for missing/null handling, and replace the odd data "null" values
#then convert to numeric.  Replace missing values with seasonal means where possible.

#--------------------------------------CLEANUP BEGIN------------------------------------------------------
#define seasons
season_groups = {
    'winter': ['_12', '_1', '_2'],
    'spring': ['_3', '_4', '_5'],
    'summer': ['_6', '_7', '_8'],
    'fall': ['_9', '_10', '_11']
}

numeric_columns = [col for col in all_sheets_df_combined.columns if col != 'business_type']
values_to_replace = ['(S)', '(NA)', '', ' ', 'nan', 'NaN']

#cleanup part 1 - imputes values seasonally
for col in numeric_columns:
    all_sheets_df_combined[col] = all_sheets_df_combined[col].replace(values_to_replace, np.nan)
    all_sheets_df_combined[col] = all_sheets_df_combined[col].astype(str).str.strip()
    all_sheets_df_combined[col] = all_sheets_df_combined[col].replace(['', 'nan', 'NaN'], np.nan)
    all_sheets_df_combined[col] = pd.to_numeric(all_sheets_df_combined[col], errors='coerce')

#apply seasonal mean
for season, months in season_groups.items():
    season_cols = [col for col in months if col in all_sheets_df_combined.columns]
    
    if season_cols:
        #calculate
        seasonal_means = all_sheets_df_combined[season_cols].mean(axis=1)
        
        #fill missing
        for col in season_cols:
            all_sheets_df_combined[col] = all_sheets_df_combined[col].fillna(seasonal_means)

#cleanup part 2 - for values that couldn't get a seasonal non-null value, cleanup those 
for col in numeric_columns:
    all_sheets_df_combined[col] = all_sheets_df_combined[col].replace(values_to_replace, np.nan)
    all_sheets_df_combined[col] = all_sheets_df_combined[col].astype(str).str.strip()
    all_sheets_df_combined[col] = all_sheets_df_combined[col].replace(['', 'nan', 'NaN'], np.nan)
    all_sheets_df_combined[col] = pd.to_numeric(all_sheets_df_combined[col], errors='coerce')
    all_sheets_df_combined[col] = all_sheets_df_combined[col].interpolate() #interpolate values that seasonal imputation missed
    all_sheets_df_combined[col] = all_sheets_df_combined[col].round().astype('Int64')
#--------------------------------------CLEANUP END------------------------------------------------------


#print(all_sheets_df_combined)
all_sheets_df_combined.to_csv(r'C:\Users\kedre\Documents\post_PPRI\MIT_xPRO_data_engineering_bootcamp\Solutions\mrtssales92-present_final_preprocessed.csv', index=False)
#all_sheets_df_combined.head()
#all_sheets_df_combined.info()


Successfully combined 30 sheets into a single dataframe.
Combined dataframe shape: (1950, 15)
